In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report, precision_recall_curve

print("1. Đang 'rã đông' mô hình và tải dữ liệu Test...")
# Load mô hình từ cuốn sổ tay .joblib
classifier = joblib.load('../models/dl_roberta_logreg.joblib')

# Load ma trận đặc trưng của tập Test
X_test = np.load('../features/deep_learning/roberta_test_features.npy')

# Load dữ liệu text test và nhãn gốc từ Kaggle
df_test = pd.read_csv('../data/interim/test_cleaned.csv') 
df_truth = pd.read_csv('../data/raw/test_labels.csv') # Đảm bảo bạn để đúng đường dẫn file gốc tải từ Kaggle

print("2. Đang cho Bác sĩ chẩn đoán (Dự đoán)...")
# Lấy xác suất (probability) thay vì nhãn cứng (0 hoặc 1) để tính ROC-AUC
y_pred_probs = classifier.predict_proba(X_test)

# Đưa kết quả dự đoán vào một DataFrame cho dễ xử lý
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
preds_df = pd.DataFrame(y_pred_probs, columns=[f"{col}_pred" for col in label_cols])
preds_df['id'] = df_test['id'].values

# Đổi tên cột nhãn gốc để phân biệt
truth_df_renamed = df_truth.copy()
for col in label_cols:
    truth_df_renamed.rename(columns={col: f"{col}_true"}, inplace=True)

print("3. Đang lọc rác Kaggle (bỏ các nhãn -1)...")
# Nối bảng dự đoán và bảng đáp án lại với nhau bằng ID
merged_df = pd.merge(truth_df_renamed, preds_df, on="id")

# LỌC RÁC: Bỏ những dòng Kaggle không chấm điểm (nhãn = -1)
clean_df = merged_df[merged_df["toxic_true"] != -1].copy()

print(f" - Số lượng comment test ban đầu: {len(merged_df)}")
print(f" - Số lượng comment test hợp lệ để chấm điểm: {len(clean_df)}\n")

print("========== KẾT QUẢ ROC-AUC (KAGGLE LEADERBOARD) ==========")
roc_auc_scores = []
for col in label_cols:
    y_true = clean_df[f"{col}_true"].values
    y_pred = clean_df[f"{col}_pred"].values
    
    score = roc_auc_score(y_true, y_pred)
    roc_auc_scores.append(score)
    print(f" - {col.capitalize():<15}: {score:.4f}")

final_score = np.mean(roc_auc_scores)
print("-" * 45)
print(f"🏆 FINAL MEAN ROC-AUC : {final_score:.4f}")
print("==========================================================")

print("========== BÁO CÁO CHI TIẾT (PRECISION, RECALL, F1, ACCURACY) ==========")

# Chuyển đổi xác suất (probability) thành nhãn cứng (0 hoặc 1) với ngưỡng 0.5
y_true_hard = clean_df[[f"{col}_true" for col in label_cols]].values

best_thresholds = {}

for col in label_cols:
    y_true = clean_df[f"{col}_true"].values
    y_prob = clean_df[f"{col}_pred"].values

    precision, recall, thresholds_pr = precision_recall_curve(y_true, y_prob)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    best_idx = np.argmax(f1)
    best_thresholds[col] = thresholds_pr[best_idx]

print("Best thresholds found:", best_thresholds)

y_pred_hard = np.zeros_like(y_true_hard, dtype=int)

for i, col in enumerate(label_cols):
    y_pred_hard[:, i] = (
        clean_df[f"{col}_pred"].values >= best_thresholds[col]
    ).astype(int)

# In báo cáo tổng hợp
report = classification_report(
    y_true_hard, 
    y_pred_hard, 
    target_names=label_cols, 
    zero_division=0
)
print(report)

# Tính riêng Accuracy tuyệt đối (Đoán đúng y chang cả 6 nhãn)
subset_acc = accuracy_score(y_true_hard, y_pred_hard)
print("-" * 45)
print(f"🎯 Exact Match Accuracy (Subset Accuracy): {subset_acc:.4f}")
print("========================================================================")

1. Đang 'rã đông' mô hình và tải dữ liệu Test...
2. Đang cho Bác sĩ chẩn đoán (Dự đoán)...
3. Đang lọc rác Kaggle (bỏ các nhãn -1)...
 - Số lượng comment test ban đầu: 153164
 - Số lượng comment test hợp lệ để chấm điểm: 63978

========== KẾT QUẢ ROC-AUC (KAGGLE LEADERBOARD) ==========
 - Toxic          : 0.9460
 - Severe_toxic   : 0.9639
 - Obscene        : 0.9546
 - Threat         : 0.9380
 - Insult         : 0.9558
 - Identity_hate  : 0.9765
---------------------------------------------
🏆 FINAL MEAN ROC-AUC : 0.9558
========== BÁO CÁO CHI TIẾT (PRECISION, RECALL, F1, ACCURACY) ==========
Best thresholds found: {'toxic': np.float32(0.9212905), 'severe_toxic': np.float32(1.0), 'obscene': np.float32(0.96705383), 'threat': np.float32(1.0), 'insult': np.float32(0.9930508), 'identity_hate': np.float32(0.24274655)}
               precision    recall  f1-score   support

        toxic       0.63      0.65      0.64      6090
 severe_toxic       0.10      0.86      0.17       367
      obsce

In [5]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report, precision_recall_curve

print("1. Đang 'rã đông' mô hình DeBERTa và tải dữ liệu Test...")
# ĐÃ ĐỔI SANG MODEL VÀ MA TRẬN CỦA DEBERTA
classifier = joblib.load('../models/dl_deberta_logreg.joblib')
X_test = np.load('../features/deep_learning/deberta_test_features.npy')

# Load dữ liệu text test và nhãn gốc từ Kaggle
df_test = pd.read_csv('../data/interim/test_cleaned.csv') 
df_truth = pd.read_csv('../data/raw/test_labels.csv')

print("2. Đang cho Bác sĩ DeBERTa chẩn đoán (Dự đoán)...")
# Lấy xác suất (probability) thay vì nhãn cứng (0 hoặc 1) để tính ROC-AUC
y_pred_probs = classifier.predict_proba(X_test)

# Đưa kết quả dự đoán vào một DataFrame cho dễ xử lý
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
preds_df = pd.DataFrame(y_pred_probs, columns=[f"{col}_pred" for col in label_cols])
preds_df['id'] = df_test['id'].values

# Đổi tên cột nhãn gốc để phân biệt
truth_df_renamed = df_truth.copy()
for col in label_cols:
    truth_df_renamed.rename(columns={col: f"{col}_true"}, inplace=True)

print("3. Đang lọc rác Kaggle (bỏ các nhãn -1)...")
# Nối bảng dự đoán và bảng đáp án lại với nhau bằng ID
merged_df = pd.merge(truth_df_renamed, preds_df, on="id")

# LỌC RÁC: Bỏ những dòng Kaggle không chấm điểm (nhãn = -1)
clean_df = merged_df[merged_df["toxic_true"] != -1].copy()

print(f" - Số lượng comment test ban đầu: {len(merged_df)}")
print(f" - Số lượng comment test hợp lệ để chấm điểm: {len(clean_df)}\n")
print("==========================================================")

print("\n========== BÁO CÁO CHI TIẾT (PRECISION, RECALL, F1, ACCURACY) ==========")

y_true_hard = clean_df[[f"{col}_true" for col in label_cols]].values

best_thresholds = {}

# TÌM NGƯỠNG TỐI ƯU CHO TỪNG NHÃN (TỐI ĐA HÓA F1)
for col in label_cols:
    y_true = clean_df[f"{col}_true"].values
    y_prob = clean_df[f"{col}_pred"].values

    precision, recall, thresholds_pr = precision_recall_curve(y_true, y_prob)
    
    # SỬA LỖI Ở ĐÂY: Cắt bỏ phần tử cuối cùng ([:-1]) để độ dài 3 mảng bằng nhau y xì đúc
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-8)

    best_idx = np.argmax(f1)
    best_thresholds[col] = thresholds_pr[best_idx]

print("💡 Ngưỡng tối ưu (Best Thresholds) tìm được:")
for col, thresh in best_thresholds.items():
    print(f"   - {col:<15}: {thresh:.4f}")

y_pred_hard = np.zeros_like(y_true_hard, dtype=int)

for i, col in enumerate(label_cols):
    y_pred_hard[:, i] = (
        clean_df[f"{col}_pred"].values >= best_thresholds[col]
    ).astype(int)

# HÀM NÀY SẼ IN RA CÁI BẢNG CHI TIẾT TỪNG NHÃN CHO BẠN ĐÂY:
report = classification_report(
    y_true_hard, 
    y_pred_hard, 
    target_names=label_cols, 
    zero_division=0
)
print(report)

# Tính riêng Accuracy tuyệt đối (Đoán đúng y chang cả 6 nhãn)
subset_acc = accuracy_score(y_true_hard, y_pred_hard)
print("-" * 45)
print(f"🎯 Exact Match Accuracy (Subset Accuracy): {subset_acc:.4f}")
print("========================================================================")

1. Đang 'rã đông' mô hình DeBERTa và tải dữ liệu Test...
2. Đang cho Bác sĩ DeBERTa chẩn đoán (Dự đoán)...
3. Đang lọc rác Kaggle (bỏ các nhãn -1)...
 - Số lượng comment test ban đầu: 153164
 - Số lượng comment test hợp lệ để chấm điểm: 63978


========== BÁO CÁO CHI TIẾT (PRECISION, RECALL, F1, ACCURACY) ==========
💡 Ngưỡng tối ưu (Best Thresholds) tìm được:
   - toxic          : 0.9327
   - severe_toxic   : 1.0000
   - obscene        : 0.9649
   - threat         : 1.0000
   - insult         : 0.9769
   - identity_hate  : 1.0000
               precision    recall  f1-score   support

        toxic       0.62      0.65      0.64      6090
 severe_toxic       0.10      0.83      0.18       367
      obscene       0.61      0.60      0.60      3691
       threat       0.04      0.91      0.08       211
       insult       0.57      0.59      0.58      3427
identity_hate       0.10      0.83      0.18       712

    micro avg       0.34      0.64      0.45     14498
    macro avg       0.